# K⁺–G4 質心距離分析

## 1. 這份 Notebook 在做什麼？

本程式計算分子動力學軌跡中，**每一顆鉀離子（預設 `resname POT`）到所有鳥嘌呤（預設 `resname GUA`）質心的距離**，並繪製距離隨模擬時間的變化。

### 執行後會得到

| 輸出檔案 | 內容 |
|---|---|
| `ion_com_distance.csv` | 每個取樣時間、離子編號與距離，可用 Excel 開啟 |
| `ion_com_distance.npz` | NumPy 格式的完整結果，供後續重新繪圖 |
| `ion_com_distance.png` | 距離－時間圖，300 dpi |
| `run_summary.txt` | 本次分析使用的檔案、selection、stride 與單位 |

## 2. 執行順序

1. 安裝套件。
2. **只修改「使用者設定區」**。
3. 依序執行「輸入檢查 → 計算 → 儲存 → 繪圖」。
4. 確認最後一格顯示所有輸出檔案的位置。

> 第一次接手時，請不要直接修改計算函數。先用一小段 DCD 測試，確認 selection 與輸出正確後，再分析完整軌跡。


## 3. 環境準備

建議使用獨立的 Conda 環境：

```bash
conda create -n ion-distance python=3.11 -y
conda activate ion-distance
pip install MDAnalysis numpy matplotlib jupyter
jupyter notebook
```

若套件已經安裝，可直接執行下一格。此 Notebook 使用：

- MDAnalysis：讀取拓撲與 DCD、選擇原子、處理週期邊界
- NumPy：整理與儲存數值
- Matplotlib：繪圖


In [ ]:
# ============================================================
# 4. 載入套件
# ============================================================

from pathlib import Path
import warnings

import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
from MDAnalysis.exceptions import NoDataError
from MDAnalysis.lib.distances import distance_array

print(f"MDAnalysis version: {mda.__version__}")
print(f"NumPy version: {np.__version__}")


## 5. 使用者設定區

一般情況只需修改下一格。

- `TOPOLOGY_FILE`：建議使用包含鍵結與原子質量資訊的 PSF；也可以使用 PDB。
- `DCD_FILES`：依模擬時間順序排列，可放一個或多個 DCD。
- `DNA_SELECTION`：用來計算質心的原子；目前代表所有 GUA 原子。
- `ION_SELECTION`：要追蹤的離子；CHARMM 中 K⁺ 常為 `POT`。
- `TIME_PER_FRAME_NS = 0.002`：NAMD timestep 2 fs、每 1000 steps 輸出一個 DCD frame。
- `STRIDE = 50`：每 50 個 frame 取一次，即每 0.1 ns 取樣一次。

> `DNA_SELECTION` 與 `ION_SELECTION` 必須依拓撲檔中的 residue name 調整，不可只看元素名稱猜測。


In [ ]:
# ============================================================
# 6. 使用者設定區：一般情況只修改這一格
# ============================================================

# ---------- 必須修改：輸入檔案 ----------
TOPOLOGY_FILE = Path("/path/to/G4_ion_center.psf")

DCD_FILES = [
    Path("/path/to/G4_ion.dcd"),
    Path("/path/to/G4_ion1000.dcd"),
]

# ---------- 必須確認：原子選擇語法 ----------
DNA_SELECTION = "resname GUA"
ION_SELECTION = "resname POT"
ION_LABEL = "K+"

# ---------- 通常不用修改：取樣與時間設定 ----------
STRIDE = 50
TIME_PER_FRAME_NS = 0.002

# 若 DNA 在軌跡中跨越週期邊界，建議設為 True。
# PSF 通常有鍵結資訊；只有 PDB 時可能無法 unwrap，程式會提出警告並繼續。
UNWRAP_DNA = True

# ---------- 輸出設定 ----------
OUTPUT_DIR = Path("./distance_results")
OUTPUT_PREFIX = "ion_com_distance"


## 7. 輸入檢查

這一格會在正式計算前檢查：

- 拓撲檔與所有 DCD 是否存在
- `STRIDE` 與時間間隔是否合理
- selection 是否真的選到 GUA 與 K⁺
- 拓撲是否提供質心計算所需的原子質量

若這格報錯，請先依錯誤訊息修正設定，不要跳過後繼續執行。


In [ ]:
# ============================================================
# 8. 檢查輸入並載入軌跡
# ============================================================

def require_existing_file(path, label):
    """確認檔案存在；若不存在，提供可直接定位設定的錯誤訊息。"""
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(
            f"找不到{label}：{path}\n"
            "請回到『使用者設定區』檢查路徑。"
        )
    return path.resolve()


if not isinstance(STRIDE, int) or STRIDE < 1:
    raise ValueError("STRIDE 必須是大於或等於 1 的整數。")

if TIME_PER_FRAME_NS <= 0:
    raise ValueError("TIME_PER_FRAME_NS 必須大於 0。")

topology_path = require_existing_file(TOPOLOGY_FILE, "拓撲檔")

if not DCD_FILES:
    raise ValueError("DCD_FILES 不可為空，請至少填入一個 DCD。")

dcd_paths = [
    require_existing_file(path, f"DCD 檔案（第 {i} 個）")
    for i, path in enumerate(DCD_FILES, start=1)
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 多個 DCD 會依 DCD_FILES 中的順序串接成一條軌跡。
universe = mda.Universe(
    str(topology_path),
    [str(path) for path in dcd_paths],
)

dna = universe.select_atoms(DNA_SELECTION)
ions = universe.select_atoms(ION_SELECTION)

if dna.n_atoms == 0:
    raise ValueError(
        f"DNA_SELECTION 沒有選到任何原子：{DNA_SELECTION!r}\n"
        "請檢查拓撲中的 resname。"
    )

if ions.n_atoms == 0:
    raise ValueError(
        f"ION_SELECTION 沒有選到任何原子：{ION_SELECTION!r}\n"
        "請檢查拓撲中的離子名稱，例如 POT 或 SOD。"
    )

# center_of_mass() 需要原子質量；PSF 通常含有完整資訊。
try:
    dna_masses = dna.masses
except NoDataError as error:
    raise ValueError(
        "拓撲缺少有效的原子質量，無法計算質心。"
        "建議改用對應的 PSF 拓撲檔。"
    ) from error

if np.any(~np.isfinite(dna_masses)) or np.any(dna_masses <= 0):
    raise ValueError(
        "DNA selection 中含有無效或小於等於 0 的原子質量，"
        "請檢查拓撲檔。"
    )

sampled_frames = len(range(0, universe.trajectory.n_frames, STRIDE))
sampling_interval_ns = STRIDE * TIME_PER_FRAME_NS

print("輸入檢查完成")
print(f"Topology: {topology_path}")
print(f"DCD files: {len(dcd_paths)}")
print(f"Total frames: {universe.trajectory.n_frames}")
print(f"Sampled frames: {sampled_frames}")
print(f"Sampling interval: {sampling_interval_ns:.3f} ns")
print(f"DNA atoms: {dna.n_atoms}")
print(f"Ions: {ions.n_atoms}")


## 9. 距離定義與計算

對每個取樣 frame：

1. 以所有符合 `DNA_SELECTION` 的原子計算 DNA 質心。
2. 計算每顆離子到該質心的距離。
3. 若軌跡含有效盒長，距離使用 minimum-image convention 處理週期邊界。

距離單位沿用 MDAnalysis 對 PDB／PSF＋DCD 的預設，通常為 **Å**。

> 本分析得到的是「離子到整體 GUA 質心的距離」，不是離子到最近 DNA 原子的距離。兩種定義不可混用。


In [ ]:
# ============================================================
# 10. 計算函數
# ============================================================

def calculate_ion_com_distances(
    universe,
    dna,
    ions,
    stride=1,
    time_per_frame_ns=0.002,
    unwrap_dna=True,
):
    """
    計算每顆離子到 DNA 質心的距離。

    Parameters
    ----------
    universe : MDAnalysis.Universe
        已載入拓撲與軌跡的 Universe。
    dna : MDAnalysis.AtomGroup
        用來計算質心的 DNA 原子群組。
    ions : MDAnalysis.AtomGroup
        要追蹤的離子原子群組；每個 atom 視為一顆離子。
    stride : int
        每隔多少個 DCD frame 取樣一次。
    time_per_frame_ns : float
        每個原始 DCD frame 對應的模擬時間，單位 ns。
    unwrap_dna : bool
        是否嘗試依鍵結資訊將 DNA 解開週期邊界。

    Returns
    -------
    result : dict[str, numpy.ndarray]
        包含 frame、time_ns、ion_index、ion_atom_index、distance_A。
        ion_index 是 selection 內從 0 開始的流水號；ion_atom_index
        是拓撲中的 atom index，兩者用途不同。
    """
    records = []
    unwrap_warning_shown = False

    for sample_number, ts in enumerate(universe.trajectory[::stride], start=1):
        if unwrap_dna:
            try:
                # 只重建 DNA；離子位置仍由 minimum-image distance 處理。
                dna.unwrap(compound="fragments")
            except (AttributeError, ValueError, NoDataError) as error:
                if not unwrap_warning_shown:
                    warnings.warn(
                        "DNA unwrap 失敗，將使用原始座標繼續計算。"
                        f"原因：{error}"
                    )
                    unwrap_warning_shown = True

        dna_com = dna.center_of_mass()

        # DCD 若有有效盒長，就使用 minimum-image convention。
        box = ts.dimensions
        use_pbc = (
            box is not None
            and len(box) == 6
            and np.all(np.isfinite(box))
            and np.all(np.asarray(box[:3]) > 0)
        )
        current_distances = distance_array(
            ions.positions,
            dna_com[np.newaxis, :],
            box=box if use_pbc else None,
        )[:, 0]

        time_ns = ts.frame * time_per_frame_ns

        for ion_index, (atom_index, distance_A) in enumerate(
            zip(ions.indices, current_distances)
        ):
            records.append(
                (
                    ts.frame,
                    time_ns,
                    ion_index,
                    atom_index,
                    distance_A,
                )
            )

        if sample_number == 1 or sample_number % 100 == 0:
            print(
                f"Processed {sample_number}/{sampled_frames} sampled frames",
                end="\r",
            )

    print(f"Processed {sampled_frames}/{sampled_frames} sampled frames")

    array = np.asarray(records, dtype=float)
    return {
        "frame": array[:, 0].astype(int),
        "time_ns": array[:, 1],
        "ion_index": array[:, 2].astype(int),
        "ion_atom_index": array[:, 3].astype(int),
        "distance_A": array[:, 4],
    }


In [ ]:
# ============================================================
# 11. 執行計算
# ============================================================

results = calculate_ion_com_distances(
    universe=universe,
    dna=dna,
    ions=ions,
    stride=STRIDE,
    time_per_frame_ns=TIME_PER_FRAME_NS,
    unwrap_dna=UNWRAP_DNA,
)

print(f"Calculated rows: {len(results['distance_A']):,}")
print(
    "Distance range: "
    f"{results['distance_A'].min():.3f}–"
    f"{results['distance_A'].max():.3f} Å"
)


## 12. 儲存數值結果

CSV 第一列是欄位名稱：

| 欄位 | 意義 | 單位 |
|---|---|---|
| `frame` | 串接後軌跡中的原始 frame 編號 | frame |
| `time_ns` | `frame × TIME_PER_FRAME_NS` | ns |
| `ion_index` | 本次 ion selection 內的流水號 | 無 |
| `ion_atom_index` | 拓撲中的 atom index | 無 |
| `distance_A` | 離子到 GUA 質心的距離 | Å |

同時保留 `ion_index` 與 `ion_atom_index`，可避免把「第幾顆被選到的離子」誤認為拓撲原子編號。


In [ ]:
# ============================================================
# 13. 儲存 CSV、NPZ 與執行摘要
# ============================================================

csv_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}.csv"
npz_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}.npz"
summary_path = OUTPUT_DIR / "run_summary.txt"

csv_data = np.column_stack(
    [
        results["frame"],
        results["time_ns"],
        results["ion_index"],
        results["ion_atom_index"],
        results["distance_A"],
    ]
)

np.savetxt(
    csv_path,
    csv_data,
    delimiter=",",
    header="frame,time_ns,ion_index,ion_atom_index,distance_A",
    comments="",
    fmt=["%d", "%.6f", "%d", "%d", "%.6f"],
)

# 計算與繪圖分離：之後可直接載入 NPZ 重畫，不必重新讀取 DCD。
np.savez_compressed(
    npz_path,
    **results,
    topology_file=str(topology_path),
    dcd_files=np.asarray([str(path) for path in dcd_paths]),
    dna_selection=DNA_SELECTION,
    ion_selection=ION_SELECTION,
    stride=STRIDE,
    time_per_frame_ns=TIME_PER_FRAME_NS,
)

summary_text = (
    f"{ION_LABEL}–G4 center-of-mass distance analysis\n"
    f"Topology: {topology_path}\n"
    f"DCD files: {len(dcd_paths)}\n"
    + "\n".join(f"  - {path}" for path in dcd_paths)
    + "\n"
    f"DNA selection: {DNA_SELECTION}\n"
    f"Ion selection: {ION_SELECTION}\n"
    f"Ion label: {ION_LABEL}\n"
    f"Total frames: {universe.trajectory.n_frames}\n"
    f"Stride: {STRIDE}\n"
    f"Time per original frame: {TIME_PER_FRAME_NS} ns\n"
    f"Sampling interval: {sampling_interval_ns} ns\n"
    f"Number of ions: {ions.n_atoms}\n"
    "Distance unit: Å\n"
)
summary_path.write_text(summary_text, encoding="utf-8")

print(f"Saved CSV: {csv_path.resolve()}")
print(f"Saved NPZ: {npz_path.resolve()}")
print(f"Saved summary: {summary_path.resolve()}")


## 14. 從 NPZ 讀取並繪圖

這一格刻意重新讀取 NPZ，而不是直接沿用記憶體中的 `results`。這可驗證儲存檔完整，也讓日後只修改圖形時不必重跑距離計算。


In [ ]:
# ============================================================
# 15. 載入 NPZ 並繪圖
# ============================================================

plot_data = np.load(npz_path)

time_ns = plot_data["time_ns"]
ion_indices = plot_data["ion_index"]
distances_A = plot_data["distance_A"]

fig, ax = plt.subplots(figsize=(14, 8))

for ion_index in np.unique(ion_indices):
    mask = ion_indices == ion_index
    ax.plot(
        time_ns[mask],
        distances_A[mask],
        linewidth=1.5,
        label=f"{ION_LABEL} ion {ion_index}",
    )

ax.set_xlabel("Time (ns)", fontsize=30)
ax.set_ylabel("Distance to GUA COM (Å)", fontsize=30)
ax.set_title(
    f"{ION_LABEL} Ion Distance to G4 Center of Mass",
    fontsize=30,
    pad=16,
)
ax.tick_params(axis="both", which="major", labelsize=24)
ax.grid(alpha=0.25)
ax.legend(fontsize=12, ncol=2, frameon=False)

fig.tight_layout()

figure_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure: {figure_path.resolve()}")


## 16. 結果檢查與常見問題

### 跑完後至少確認四件事

1. `Ions` 數量是否符合系統實際的 K⁺ 數量。
2. `Sampling interval` 是否符合設定；目前 `50 × 0.002 = 0.1 ns`。
3. 圖的時間範圍是否等於預期模擬長度。
4. 距離曲線是否有不合理的瞬間跳動；若有，先檢查 PBC、DCD 順序與 topology 是否匹配。

### 常見錯誤

**`FileNotFoundError`**  
回到使用者設定區修正路徑。Linux 路徑區分大小寫。

**selection 沒有選到原子**  
用 VMD 或 MDAnalysis 檢查拓撲中的 residue name。K⁺ 不一定都叫 `POT`，Na⁺ 常見名稱為 `SOD`。

**`unwrap` 警告**  
PDB 可能缺乏鍵結資訊。優先使用模擬時對應的 PSF；若已確認 G4 沒有跨越盒子，也可將 `UNWRAP_DNA = False`。

**多個 DCD 的時間不連續**  
`DCD_FILES` 必須依模擬先後順序排列，而且所有 DCD 必須和同一個 topology 相符。

**想改成 Na⁺**  
先確認拓撲中的 Na⁺ residue name，再將 `ION_SELECTION` 改成例如 `"resname SOD"`；圖例文字也應同步改成 Na⁺。

### 交接時不可省略的研究定義

- 本 Notebook 分析的是 **ion–GUA COM distance**。
- 距離單位為 Å，時間單位為 ns。
- `TIME_PER_FRAME_NS = 0.002` 只適用於 timestep 2 fs 且每 1000 steps 輸出 DCD 的軌跡。
- 改變 selection、stride 或時間換算後，必須記錄在研究方法與 `run_summary.txt`。
